# Notebook 03 of 7 — Basket X-Ray + Risk (Track B: Free-only)

*Portfolio Intelligence Engine — User Guide Series, **Track B (free sources only)**.*
[Series README](../portfolio/README.md) · [Story Bible](../portfolio/STORY_BIBLE.md) · Epic [#1428](https://github.com/prajoria/OpenBB/issues/1428) · This notebook [#1439](https://github.com/prajoria/OpenBB/issues/1439) · Track A counterpart: [`../portfolio/03-basket-xray-and-risk.ipynb`](../portfolio/03-basket-xray-and-risk.ipynb).

---

## Where we are in Sam's story

This is the free-only mirror of NB03 — the pivot chapter. NB02 built an opinion on a single name; now I want the basket-level picture. The question I've been dodging: what do all 10 positions look like *together* once I stop pretending each ETF is one thing?

By the end of the notebook we answer one question:

> *What am I actually exposed to at the basket level, after looking through my ETFs — using ONLY free-authoritative sources?*

Track A NB03 gets the naive-vs-x-ray delta from N-PORT + `fmp_cached` sector profiles. Track B uses the SAME N-PORT look-through (SEC is the provider-of-record for N-PORT either way) but replaces `fmp_cached` sector labels with a hand-curated free-tier sector map for the top mega-caps and an honest "Unknown" bucket for the tail. The numeric x-ray moves in the same direction as Track A; the exact percentages differ by how the Unknown tail bucket is drawn.


## 0. Before we run anything

Same venv rule as every notebook in this series — `.venv_portfolio`, or later cells crash with mismatched extensions. State goes into `.notebook_state/` (gitignored). We write to `xray_free.pkl` (NB04/05/06 Track B pick it up) — do NOT overwrite the Track A `xray.pkl`.


In [ ]:
# [Track B / NB03 §0] environment sanity — assert .venv_portfolio + STATE dir
import sys, pathlib

assert "venv_portfolio" in sys.executable, (
    "Portfolio notebooks require .venv_portfolio, not the current interpreter.\n"
    "See NB01 §0 for setup.\n"
    f"Currently running: {sys.executable}"
)
STATE = pathlib.Path(".notebook_state")
STATE.mkdir(exist_ok=True)
print(f"Python:                   {sys.version.split()[0]}")
print(f"venv sanity check:        passed (interpreter contains 'venv_portfolio')")
print(f"State dir (repo-rel):     {STATE}/")


Python:                   3.12.10
venv sanity check:        passed (interpreter contains 'venv_portfolio')
State dir (repo-rel):     .notebook_state/


## 0.5 Why free-only NB03 looks different from Track A

The look-through math is identical — SEC Form N-PORT is *the* filing every '40-Act US-registered fund files, so it's free-authoritative for both tracks. Track A calls `obb.etf.nport_disclosure(provider="sec")`; Track B calls the same endpoint via the shared [`_nport_lookthrough.py`](../portfolio/_nport_lookthrough.py) helper.

Where the tracks diverge is *sector classification for constituents*. Track A rolls each ETF's N-PORT holdings up by sector using `obb.equity.profile(provider="fmp_cached")`. Free-tier can't do that — there is no free-authoritative ticker→sector API that covers the whole US equity universe cleanly. Track B does the honest thing:

1. **Hand-curated sector map** for the top ~40 mega-caps that dominate    the QQQ / VTI / VNQ tops (MSFT, NVDA, AAPL, GOOGL, AMZN, etc.).    Sourced from public 10-K SIC codes and GICS classifications; safe    to hard-code because these companies don't switch sector.
2. **Bucket the tail as `Unknown`** — approximately 15-20% of x-ray    weight lands here (compared with Track A's 16% "Other (small)").    The rollup transparency line names the exact number.
3. **SIC → GICS mapping gap** is documented in "What is NOT." A future    Track B upgrade could parse SIC codes from SEC XBRL company facts,    but SIC ≠ GICS and the mapping is lossy.

Provider chain for this notebook:

| Data path | Provider (Track B) |
|---|---|
| ETF constituent look-through | SEC N-PORT (`sec`) via `_nport_lookthrough` |
| Prices / returns for risk | CBOE EOD (`cboe`) |
| Sector classification | Hand-curated free-tier map + Unknown bucket |
| Benchmark (SPY) for beta/tracking | CBOE EOD (`cboe`) |

Bare-term pointer: portfolio, position, weight, diversification, sector breakdown, look-through, HHI, Sharpe, volatility, maximum drawdown, tracking error, benchmark, risk-free rate, correlation were all cited with Investopedia links in Track A NB03. This notebook does not re-cite them.


## 1. Load the basket

The 10-position basket from NB01 (Track B shares the same basket file with Track A). We load `.notebook_state/basket.json` if it exists; otherwise regenerate from the locked list. DO NOT overwrite basket.json — both tracks read from the same file.


In [ ]:
# [Track B / NB03 §1] Load the shared 10-position basket (do NOT modify)
import json
from pathlib import Path

state = Path(".notebook_state")
basket_path = state / "basket.json"

BASKET_LOCKED = [
    {"symbol": "MSFT",  "weight": 0.12, "kind": "equity"},
    {"symbol": "NVDA",  "weight": 0.10, "kind": "equity"},
    {"symbol": "GOOGL", "weight": 0.08, "kind": "equity"},
    {"symbol": "AAPL",  "weight": 0.08, "kind": "equity"},
    {"symbol": "AMD",   "weight": 0.06, "kind": "equity"},
    {"symbol": "QQQ",   "weight": 0.15, "kind": "etf"},
    {"symbol": "VTI",   "weight": 0.20, "kind": "etf"},
    {"symbol": "VNQ",   "weight": 0.08, "kind": "etf"},
    {"symbol": "BND",   "weight": 0.10, "kind": "etf"},
    {"symbol": "GLD",   "weight": 0.03, "kind": "etf"},
]

if basket_path.exists():
    basket = json.loads(basket_path.read_text(encoding="utf-8"))
    for row in basket:
        if "ticker" in row and "symbol" not in row:
            row["symbol"] = row.pop("ticker")
    print(f"Loaded basket from {basket_path} (shared with Track A — not modified)")
else:
    basket = BASKET_LOCKED
    state.mkdir(exist_ok=True)
    basket_path.write_text(json.dumps(basket, indent=2), encoding="utf-8")
    print(f"Regenerated basket from locked list -> {basket_path}")

total_w = sum(p["weight"] for p in basket)
print(f"Positions: {len(basket)}    Total weight: {total_w*100:.1f}%")
print()
print(f"{'Symbol':<8}{'Weight':>8}   Kind")
print("-" * 40)
for p in basket:
    print(f"{p['symbol']:<8}{p['weight']*100:>7.1f}%   {p.get('kind', '?')}")


Loaded basket from .notebook_state\basket.json (shared with Track A — not modified)
Positions: 10    Total weight: 100.0%

Symbol    Weight   Kind
----------------------------------------
MSFT       12.0%   equity
NVDA       10.0%   equity
GOOGL       8.0%   equity
AAPL        8.0%   equity
AMD         6.0%   equity
QQQ        15.0%   etf
VTI        20.0%   etf
VNQ         8.0%   etf
BND        10.0%   etf
GLD         3.0%   etf


## 2. Naive sector view (free-tier, no look-through)

Before the pivot, the naive spreadsheet view: each ETF gets its own sector bucket, each single-name equity gets a hand-labeled sector. Track A queries `fmp_cached` for equity sectors; Track B uses the hand-curated free-tier map. For the 5 single-name equities in this basket (MSFT, NVDA, GOOGL, AAPL, AMD) the mapping is unambiguous. Hold the naive Tech percentage — we'll compare it against the x-ray in §4.


In [ ]:
# [Track B / NB03 §2] Naive sector view — each ETF is its own row, no paid provider
# Free-tier sector map for the 5 single-name equities in the basket +
# the ETFs' "obvious" hand-labels. No provider calls in this cell.
NAIVE_SECTOR_LABELS = {
    "MSFT":  "Technology",
    "NVDA":  "Technology",
    "AAPL":  "Technology",
    "AMD":   "Technology",
    "GOOGL": "Communication Services",
    "QQQ":   "Tech ETF",
    "VTI":   "Broad ETF",
    "VNQ":   "REIT",
    "BND":   "Bond Fund",
    "GLD":   "Commodity",
}

naive_by_sector = {}
for pos in basket:
    sym = pos["symbol"]; w = pos["weight"]
    sec = NAIVE_SECTOR_LABELS.get(sym, "Unknown")
    naive_by_sector[sec] = naive_by_sector.get(sec, 0.0) + w

print("Naive sector view (each ETF is its own bucket, no look-through):")
print(f"{'Sector':<24}{'Weight':>10}")
print("-" * 36)
for sector, w in sorted(naive_by_sector.items(), key=lambda kv: -kv[1]):
    print(f"{sector:<24}{w*100:>9.1f}%")
tech_naive = naive_by_sector.get("Technology", 0.0) + naive_by_sector.get("Tech ETF", 0.0)
print()
print(f"Raw Tech (equity 'Technology' + 'Tech ETF' bucket): {tech_naive*100:.1f}%")
print("Hold that number.")


Naive sector view (each ETF is its own bucket, no look-through):
Sector                      Weight
------------------------------------
Technology                   36.0%
Broad ETF                    20.0%
Tech ETF                     15.0%
Bond Fund                    10.0%
Communication Services        8.0%
REIT                          8.0%
Commodity                     3.0%

Raw Tech (equity 'Technology' + 'Tech ETF' bucket): 51.0%
Hold that number.


## 3. The x-ray — SEC N-PORT look-through

Same call Track A NB03 makes: read each fund's Form N-PORT disclosure via `obb.etf.nport_disclosure(provider="sec")`, re-weight each underlying issuer by (basket weight × ETF weight-in-basket). N-PORT is quarterly-visible with ~30-60 days lag — for a concentration story that's a rounding error.

GLD is a commodity grantor trust; it files 10-K/8-K, not N-PORT. The helper raises `NportUnavailable` and we pass GLD through as opaque. This is the exact same behavior as Track A NB03 — there is no "free tier" difference in the look-through math itself.


In [ ]:
# [Track B / NB03 §3] N-PORT look-through via shared _nport_lookthrough helper
# The helper is repo-shared with Track A (do not modify from Track B).
import sys, pathlib
sys.path.insert(0, str(pathlib.Path("../portfolio").resolve()))
from _nport_lookthrough import effective_positions, NportUnavailable  # noqa: E402

EQUITY_ETFS = {"QQQ", "VTI", "VNQ", "SPY", "DIA", "IWM", "VOO", "VEA", "VXUS", "VWO"}
BOND_ETFS = {"BND", "AGG", "TLT", "SHY", "TIP", "SCHP"}
COMMODITY_TRUSTS = {"GLD", "SLV", "DBC"}

effective, non_nport, opaque = effective_positions(
    basket,
    equity_etfs=EQUITY_ETFS,
    bond_etfs=BOND_ETFS,
    commodity_trusts=COMMODITY_TRUSTS,
    top_n_per_etf=50,
)

print(f"Basket: {len(basket)} positions")
print(f"After N-PORT look-through: {len(effective)} distinct effective positions")
print(f"Total effective weight: {sum(effective.values())*100:.1f}%")
print()
if opaque:
    print("Opaque (no look-through possible):")
    for sym, reason in opaque:
        print(f"  {sym}: {reason[:80]}")
    print()

print("Top 15 effective positions (post-look-through):")
print(f"{'Issuer / bucket':<40}{'Weight':>10}")
print("-" * 52)
for key, w in sorted(effective.items(), key=lambda kv: -kv[1])[:15]:
    print(f"{str(key)[:38]:<40}{w*100:>9.2f}%")


Basket: 10 positions
After N-PORT look-through: 158 distinct effective positions
Total effective weight: 100.0%

Opaque (no look-through possible):
  GLD: commodity trust — no look-through

Top 15 effective positions (post-look-through):
Issuer / bucket                             Weight
----------------------------------------------------
MSFT                                        12.00%
NVDA                                        10.00%
TAIL_VTI                                     9.32%
TAIL_BND                                     8.37%
GOOGL                                        8.00%
AAPL                                         8.00%
AMD                                          6.00%
GLD                                          3.00%
TAIL_QQQ                                     1.99%
BOND_BND                                     1.60%
NVIDIA Corp.                                 1.30%
NVIDIA Corp                                  1.28%
Apple Inc                                    1

## 4. Sector view WITH look-through — the free-tier pivot

Same chart as §2, but on the flattened basket. Instead of Track A's `fmp_cached` sector profiles, Track B uses a hand-curated sector map for the top mega-caps that actually dominate QQQ / VTI / VNQ weights, with the tail bucketed as `Unknown` (honestly, not silently).

Compare against Track A's numbers: Tech goes from ~51% naive to ~46% x-ray in Track A. In Track B the Tech line lands close to that same level, but the tail Unknown bucket grabs more weight than Track A's "Other (small)" because our free-tier map doesn't classify the small issuers in the tail. That gap is the honest cost of free-tier here.


In [ ]:
# [Track B / NB03 §4] Sector view WITH look-through — free-tier sector map
# NO paid-provider calls. Uses hand-curated FREE_SECTOR_MAP for the top
# ~40 mega-caps that dominate QQQ/VTI/VNQ weights; everything else
# lands in "Unknown" (honest name for what free-tier can't resolve).
import sys, pathlib
sys.path.insert(0, str(pathlib.Path("../portfolio").resolve()))
from _nport_lookthrough import nport_holdings, NportUnavailable  # noqa: E402

# Free-tier sector map — GICS-aligned; sourced from public 10-K SIC codes.
# Keys are the issuer 'name' as it appears on N-PORT rows (with common
# spelling variants). This is deliberately small: it's the top-of-QQQ/VTI
# universe that carries >90% of the weight after look-through.
FREE_SECTOR_MAP = {
    # Technology
    "Microsoft Corp.":                     "Technology",
    "Microsoft Corp":                      "Technology",
    "NVIDIA Corp.":                        "Technology",
    "NVIDIA Corp":                         "Technology",
    "Apple Inc.":                          "Technology",
    "Apple Inc":                           "Technology",
    "Broadcom Inc.":                       "Technology",
    "Broadcom Inc":                        "Technology",
    "Advanced Micro Devices, Inc.":        "Technology",
    "Advanced Micro Devices, Inc":         "Technology",
    "Cisco Systems, Inc.":                 "Technology",
    "Applied Materials, Inc.":             "Technology",
    "Lam Research Corp.":                  "Technology",
    "KLA Corp.":                           "Technology",
    "Intel Corp.":                         "Technology",
    "Texas Instruments Inc.":              "Technology",
    "Analog Devices, Inc.":                "Technology",
    "Oracle Corp.":                        "Technology",
    "Salesforce, Inc.":                    "Technology",
    "Adobe Inc.":                          "Technology",
    "ServiceNow, Inc.":                    "Technology",
    "Palantir Technologies Inc.":          "Technology",
    "Micron Technology, Inc.":             "Technology",
    "International Business Machines Corp.": "Technology",
    "QUALCOMM Inc.":                       "Technology",
    "Intuit Inc.":                         "Technology",
    # Communication Services
    "Alphabet Inc.":                       "Communication Services",
    "Alphabet Inc":                        "Communication Services",
    "Meta Platforms, Inc.":                "Communication Services",
    "Netflix, Inc.":                       "Communication Services",
    "T-Mobile US, Inc.":                   "Communication Services",
    "Comcast Corp.":                       "Communication Services",
    "Verizon Communications Inc.":         "Communication Services",
    "AT&T Inc.":                           "Communication Services",
    # Consumer Cyclical
    "Amazon.com, Inc.":                    "Consumer Cyclical",
    "Tesla, Inc.":                         "Consumer Cyclical",
    "Home Depot, Inc.":                    "Consumer Cyclical",
    "McDonald's Corp.":                    "Consumer Cyclical",
    "Booking Holdings Inc.":               "Consumer Cyclical",
    # Consumer Defensive
    "Walmart Inc.":                        "Consumer Defensive",
    "Costco Wholesale Corp.":              "Consumer Defensive",
    "PepsiCo, Inc.":                       "Consumer Defensive",
    "The Coca-Cola Co.":                   "Consumer Defensive",
    "Procter & Gamble Co.":                "Consumer Defensive",
    # Financial Services
    "JPMorgan Chase & Co.":                "Financial Services",
    "Berkshire Hathaway Inc.":             "Financial Services",
    "Visa Inc.":                           "Financial Services",
    "Mastercard Inc.":                     "Financial Services",
    "Bank of America Corp.":               "Financial Services",
    # Healthcare
    "UnitedHealth Group Inc.":             "Healthcare",
    "Eli Lilly and Co.":                   "Healthcare",
    "Johnson & Johnson":                   "Healthcare",
    "AbbVie Inc.":                         "Healthcare",
    "Merck & Co., Inc.":                   "Healthcare",
    "Amgen Inc.":                          "Healthcare",
    "Gilead Sciences, Inc.":               "Healthcare",
    "Intuitive Surgical, Inc.":            "Healthcare",
    # Energy
    "Exxon Mobil Corp.":                   "Energy",
    "Chevron Corp.":                       "Energy",
    # Industrials
    "Honeywell International Inc.":        "Industrials",
    "Linde PLC":                           "Basic Materials",
}

def _rollup_free(rows, weight_scale=1.0, top_n=50):
    """Free-tier sector rollup — free-tier only; uses FREE_SECTOR_MAP + Unknown."""
    out = {}
    counts = {}
    for i, r in enumerate(sorted(rows, key=lambda x: -(x.get("weight") or 0.0))):
        w = (r.get("weight") or 0.0) * weight_scale
        if w <= 0:
            continue
        asset_cat = r.get("asset_category") or ""
        if asset_cat in ("DBT", "ABS-APCP", "ABS-CBDO", "ABS-MBS", "ABS-O",
                         "ABS-CDO", "ABS-CMBS", "ABS-RMBS", "LOAN"):
            bucket = "Fixed Income"
        elif asset_cat == "STIV":
            bucket = "Short-Term / Cash"
        elif asset_cat == "RE":
            bucket = "Real Estate"
        elif asset_cat == "COMM":
            bucket = "Commodity"
        elif i < top_n and r.get("name") in FREE_SECTOR_MAP:
            bucket = FREE_SECTOR_MAP[r["name"]]
        else:
            bucket = "Unknown"
        out[bucket] = out.get(bucket, 0.0) + w
        counts[bucket] = counts.get(bucket, 0) + 1
    return out, counts

xray_by_sector = {}
unknown_count = 0
for pos in basket:
    sym = pos["symbol"]; w = pos["weight"]
    if sym in COMMODITY_TRUSTS:
        xray_by_sector["Commodity"] = xray_by_sector.get("Commodity", 0.0) + w
        continue
    if sym in BOND_ETFS:
        xray_by_sector["Fixed Income"] = xray_by_sector.get("Fixed Income", 0.0) + w
        continue
    if sym in EQUITY_ETFS:
        try:
            rows = nport_holdings(sym)
        except NportUnavailable:
            xray_by_sector[sym] = xray_by_sector.get(sym, 0.0) + w
            continue
        sec_w, cnt = _rollup_free(rows, weight_scale=w, top_n=50)
        for k, v in sec_w.items():
            xray_by_sector[k] = xray_by_sector.get(k, 0.0) + v
        unknown_count += cnt.get("Unknown", 0)
    else:
        sec = NAIVE_SECTOR_LABELS.get(sym, "Unknown")
        xray_by_sector[sec] = xray_by_sector.get(sec, 0.0) + w

print("Look-through sector view (N-PORT + free-tier sector map):")
print(f"{'Sector':<24}{'Weight':>10}")
print("-" * 36)
for sector, w in sorted(xray_by_sector.items(), key=lambda kv: -kv[1]):
    print(f"{sector:<24}{w*100:>9.2f}%")

print()
print("Side-by-side delta (naive vs N-PORT look-through, free-tier):")
print(f"{'Sector':<24}{'Naive':>10}{'X-Ray':>10}{'Delta':>10}")
print("-" * 56)
for sec in sorted(set(naive_by_sector) | set(xray_by_sector),
                  key=lambda s: -xray_by_sector.get(s, 0)):
    n = naive_by_sector.get(sec, 0.0) * 100
    x = xray_by_sector.get(sec, 0.0) * 100
    print(f"{sec:<24}{n:>9.1f}%{x:>9.2f}%{x-n:>+9.1f}%")

tech_xray = xray_by_sector.get("Technology", 0.0) + xray_by_sector.get("Tech ETF", 0.0)
print()
print(f"Effective Tech exposure: {tech_xray*100:.1f}%  "
      f"(was {tech_naive*100:.1f}% naive — N-PORT look-through redistributes)")
print(f"Unknown-bucket size:     {xray_by_sector.get('Unknown', 0.0)*100:.1f}%  "
      f"(constituents outside the free-tier sector map — documented gap)")
print(f"Rollup transparency:     {unknown_count} constituent rows fell into Unknown.")


Look-through sector view (N-PORT + free-tier sector map):
Sector                      Weight
------------------------------------
Technology                  46.10%
Unknown                     26.23%
Communication Services      11.10%
Fixed Income                10.00%
Commodity                    3.00%
Consumer Cyclical            1.37%
Consumer Defensive           1.07%
Healthcare                   0.63%
Basic Materials              0.27%
Short-Term / Cash            0.16%
Industrials                  0.12%

Side-by-side delta (naive vs N-PORT look-through, free-tier):
Sector                       Naive     X-Ray     Delta
--------------------------------------------------------
Technology                   36.0%    46.10%    +10.1%
Unknown                       0.0%    26.23%    +26.2%
Communication Services        8.0%    11.10%     +3.1%
Fixed Income                  0.0%    10.00%    +10.0%
Commodity                     3.0%     3.00%     +0.0%
Consumer Cyclical             0.0% 

## 5. HHI + Effective-N — the two numbers to quote

Same math as Track A, no provider dependency — both metrics are pure arithmetic on the weights we already have.

- **HHI** = sum of squared weights. Higher = more concentrated. DOJ   antitrust threshold for "unconcentrated" is 0.15 (1500).
- **Effective-N** = 1 / HHI. "How many equal-weight positions am I   really holding?"

Prediction from the story: HHI drops post-look-through because QQQ / VTI spread the ETF weight across many small positions, so "3 big ETF bets" is really 40+ tiny slivers of well-known names. Track A saw HHI 0.1206 → 0.0598, N 8.29 → 16.72. Track B should land in the same ballpark since the effective-position weights come from the same N-PORT source.


In [ ]:
# [Track B / NB03 §5] HHI + Effective-N — pure arithmetic on effective weights
def _hhi(weights):
    return sum(w * w for w in weights)

raw_weights = [p["weight"] for p in basket]
hhi_raw = _hhi(raw_weights)
neff_raw = 1.0 / hhi_raw if hhi_raw > 0 else float("nan")

xray_weights = list(effective.values())
hhi_xray = _hhi(xray_weights)
neff_xray = 1.0 / hhi_xray if hhi_xray > 0 else float("nan")

print(f"{'Metric':<24}{'Raw':>12}{'X-Ray':>12}{'Delta':>12}")
print("-" * 60)
print(f"{'HHI':<24}{hhi_raw:>12.4f}{hhi_xray:>12.4f}{hhi_xray-hhi_raw:>+12.4f}")
print(f"{'Effective-N':<24}{neff_raw:>12.2f}{neff_xray:>12.2f}{neff_xray-neff_raw:>+12.2f}")
print()
print(f"I hold {len(basket)} tickers. Effective-N under look-through is {neff_xray:.1f} —")
print(f"my concentration is equivalent to holding {neff_xray:.1f} equal-weight positions,")
print(f"not the {len(basket)} I thought I did.")


Metric                           Raw       X-Ray       Delta
------------------------------------------------------------
HHI                           0.1206      0.0598     -0.0608
Effective-N                     8.29       16.72       +8.43

I hold 10 tickers. Effective-N under look-through is 16.7 —
my concentration is equivalent to holding 16.7 equal-weight positions,
not the 10 I thought I did.


## 6. Risk metrics via CBOE EOD prices

Sharpe, annualized volatility, max drawdown, and tracking error vs SPY — the four seatbelts. Track A pulls prices via `fmp_cached`; Track B pulls the same shape from `obb.equity.price.historical(provider="cboe")`. CBOE is the listing-exchange EOD source: free, authoritative, and sufficient for ~3 months of daily returns.

We compute the metrics directly (no `portfolio_intel.risk.metrics` call) because the router will attempt to fetch dividends via the same provider, and only some providers support that cleanly. Direct math on returns keeps the free-tier path honest.


In [ ]:
# [Track B / NB03 §6] Risk metrics from CBOE EOD prices — direct computation
import time
import math
import numpy as np
from openbb import obb
import warnings; warnings.filterwarnings("ignore")

START = "2026-05-01"
END = "2026-07-24"
symbols_to_fetch = [p["symbol"] for p in basket] + ["SPY"]

def _daily_returns_cboe(sym):
    try:
        df = obb.equity.price.historical(
            symbol=sym, provider="cboe", start_date=START, end_date=END
        ).to_df()
    except Exception:
        return []
    if df.empty or "close" not in df.columns:
        return []
    closes = df["close"].dropna().tolist()
    if len(closes) < 2:
        return []
    return list(np.diff(closes) / closes[:-1])

t0 = time.perf_counter()
returns_source = {}
missing = []
for sym in symbols_to_fetch:
    r = _daily_returns_cboe(sym)
    if r:
        returns_source[sym] = r
    else:
        missing.append(sym)
print(f"CBOE historical: fetched {len(returns_source)}/{len(symbols_to_fetch)} symbols "
      f"in {time.perf_counter()-t0:.1f}s")
if missing:
    print(f"  (missing: {missing})")

bench = returns_source.pop("SPY", [])
if returns_source and bench:
    common_len = min(min(len(v) for v in returns_source.values()), len(bench))
    returns_source = {k: v[:common_len] for k, v in returns_source.items()}
    bench = bench[:common_len]

    basket_for_risk = [p for p in basket if p["symbol"] in returns_source]
    wt = sum(p["weight"] for p in basket_for_risk)
    basket_for_risk = [{"symbol": p["symbol"], "weight": p["weight"] / wt} for p in basket_for_risk]

    # Weighted portfolio daily returns
    weights = np.array([p["weight"] for p in basket_for_risk])
    ret_matrix = np.array([returns_source[p["symbol"]] for p in basket_for_risk])
    port_ret = weights @ ret_matrix
    bench_ret = np.array(bench)

    # Metrics (annualized where sensible, 252 trading days)
    vol_ann = float(np.std(port_ret, ddof=1) * math.sqrt(252))
    mean_ann = float(np.mean(port_ret) * 252)
    rf = 0.045  # ~risk-free proxy (T-bill); free-tier assumption, printed for the reader
    sharpe = (mean_ann - rf) / vol_ann if vol_ann > 0 else float("nan")

    # Max drawdown from cumulative wealth curve
    wealth = np.cumprod(1 + port_ret)
    running_peak = np.maximum.accumulate(wealth)
    dd = wealth / running_peak - 1.0
    max_dd = float(dd.min())

    # Tracking error vs SPY
    diff = port_ret - bench_ret
    tracking_err = float(np.std(diff, ddof=1) * math.sqrt(252))

    # Beta vs SPY
    cov = np.cov(port_ret, bench_ret, ddof=1)[0, 1]
    bench_var = float(np.var(bench_ret, ddof=1))
    beta = cov / bench_var if bench_var > 0 else float("nan")

    print()
    print(f"Portfolio-level risk (CBOE EOD, {common_len} daily returns, {len(basket_for_risk)} positions):")
    print(f"  Annualized volatility: {vol_ann*100:>7.2f}%")
    print(f"  Annualized return:     {mean_ann*100:>7.2f}%")
    print(f"  Sharpe (rf={rf:.3f}):     {sharpe:>7.2f}")
    print(f"  Max drawdown:          {max_dd*100:>7.2f}%")
    print(f"  Tracking error vs SPY: {tracking_err*100:>7.2f}%")
    print(f"  Beta vs SPY:           {beta:>7.2f}")
    risk_ok = True
else:
    print("Cannot compute risk metrics — CBOE returns unavailable.")
    vol_ann = mean_ann = sharpe = max_dd = tracking_err = beta = float("nan")
    basket_for_risk = []
    risk_ok = False


CBOE historical: fetched 11/11 symbols in 5.9s

Portfolio-level risk (CBOE EOD, 56 daily returns, 10 positions):
  Annualized volatility:   17.31%
  Annualized return:       14.61%
  Sharpe (rf=0.045):        0.58
  Max drawdown:            -6.45%
  Tracking error vs SPY:    6.82%
  Beta vs SPY:              1.20


## 7. Concentration — top-K weights + single-name kill-shot

Rounds out the picture: the top-1, top-5, top-10 effective weights and the largest-single-position kill-shot (portfolio hit if the top effective position drops 20%). Pure arithmetic on `effective` from §3 — no provider dependency, so free-tier and paid-tier produce the same answer here.


In [ ]:
# [Track B / NB03 §7] Concentration — top-K + kill-shot
def _topk(d, k):
    return sum(sorted(d.values(), reverse=True)[:k])

top1 = _topk(effective, 1)
top5 = _topk(effective, 5)
top10 = _topk(effective, 10)

print("Concentration (post-look-through):")
print(f"  {'HHI':<24}{hhi_xray:>10.4f}")
print(f"  {'Effective-N':<24}{neff_xray:>10.2f}")
print(f"  {'Top-1 weight':<24}{top1*100:>9.2f}%")
print(f"  {'Top-5 weight':<24}{top5*100:>9.2f}%")
print(f"  {'Top-10 weight':<24}{top10*100:>9.2f}%")

top_sym, top_w = max(effective.items(), key=lambda kv: kv[1])
kill = top_w * 0.20
print()
print(f"Single-name kill-shot (20% drawdown on largest effective position):")
print(f"  Largest effective position: {top_sym} at {top_w*100:.2f}%")
print(f"  If {top_sym} drops 20%: portfolio hit = {kill*100:.2f}%")


Concentration (post-look-through):
  HHI                         0.0598
  Effective-N                  16.72
  Top-1 weight                12.00%
  Top-5 weight                47.69%
  Top-10 weight               68.28%

Single-name kill-shot (20% drawdown on largest effective position):
  Largest effective position: MSFT at 12.00%
  If MSFT drops 20%: portfolio hit = 2.40%


## 8. Silent-failure guard — reader-facing example

One of the core disciplines: an empty result from non-empty input is almost always a bug. Cell below demonstrates the guard on a pathological input — an empty basket — and shows we get a *loud* zero (log line + explanatory output) rather than a silent zero risk number that reads like "portfolio is safe."


In [ ]:
# [Track B / NB03 §8] Silent-failure guard — empty basket must be loud
empty_basket = []
empty_weights = [p["weight"] for p in empty_basket]
empty_hhi = _hhi(empty_weights)
if len(empty_basket) == 0:
    print("GUARD FIRED: basket has 0 positions; HHI is undefined, not zero.")
    print("  A quiet '0.0' here would be indistinguishable from a perfectly")
    print("  diversified book — exactly the failure mode the discipline forbids.")
    empty_neff = float('nan')
else:
    empty_neff = 1.0 / empty_hhi if empty_hhi > 0 else float('nan')
print(f"empty_hhi={empty_hhi}   empty_neff={empty_neff}")


GUARD FIRED: basket has 0 positions; HHI is undefined, not zero.
  A quiet '0.0' here would be indistinguishable from a perfectly
  diversified book — exactly the failure mode the discipline forbids.
empty_hhi=0   empty_neff=nan


## 9. Save state for NB04 / NB05 / NB06 (Track B)

Track B downstream picks up:
- `.notebook_state/basket.json` — shared with Track A (not modified here)
- `.notebook_state/xray_free.pkl` — Track B x-ray artifact (SEPARATE from Track A's `xray.pkl` so both tracks can be re-run without stomping)
- Risk metrics go into the same pickle keyed as `risk_metrics_free`

**Do not overwrite `xray.pkl`.** Track A NB04/05/06 read that file; Track B siblings will read `xray_free.pkl`.


In [ ]:
# [Track B / NB03 §9] Save Track-B state — write ONLY to xray_free.pkl
import pickle  # noqa: S403 — trusted local artifact under .notebook_state/
from pathlib import Path

state = Path(".notebook_state")
state.mkdir(exist_ok=True)

artifact = {
    "track": "B (free-only)",
    "basket": basket,
    "effective_positions": effective,
    "naive_by_sector": naive_by_sector,
    "xray_by_sector": xray_by_sector,
    "hhi_raw": hhi_raw,
    "hhi_xray": hhi_xray,
    "neff_raw": neff_raw,
    "neff_xray": neff_xray,
    "top1_weight": top1,
    "top5_weight": top5,
    "top10_weight": top10,
    "risk_metrics_free": {
        "provider": "cboe",
        "window": {"start": START, "end": END},
        "annualized_volatility": vol_ann,
        "annualized_return": mean_ann,
        "sharpe": sharpe,
        "max_drawdown": max_dd,
        "tracking_error_vs_spy": tracking_err,
        "beta_vs_spy": beta,
        "basket_for_risk": basket_for_risk,
        "note": "Direct computation; free-tier only.",
    },
    "note": "Track B free-tier x-ray. Do not confuse with Track A xray.pkl.",
}

out = state / "xray_free.pkl"
out.write_bytes(pickle.dumps(artifact))
print(f"Wrote (repo-rel):  {out}    {out.stat().st_size:,} bytes")
print(f"Track A's xray.pkl NOT modified by this notebook.")


Wrote (repo-rel):  .notebook_state\xray_free.pkl    6,675 bytes
Track A's xray.pkl NOT modified by this notebook.


---

## What is NOT in this notebook

- **Cash / short positions.** Same gap as Track A NB03 — basket is   long-only equity + ETFs. Cash-as-position is [#903](https://github.com/prajoria/OpenBB/issues/903),   shorts are [#904](https://github.com/prajoria/OpenBB/issues/904).
- **Factor decomposition (Fama-French, Carhart).** `openbb_famafrench`   exists but isn't wired into portfolio_intel risk.
- **Perfect sector coverage.** The free-tier `FREE_SECTOR_MAP` covers   the top ~60 mega-caps that carry >80% of the QQQ / VTI / VNQ weight,   but the tail (~15-20% of x-ray weight) lands in `Unknown`. Track A   routes the tail through `fmp_cached` sector profiles; Track B names   the gap honestly rather than fake-classifying via yfinance-recorded   data. A future upgrade could parse SEC XBRL SIC codes and map SIC   → GICS — but SIC and GICS are not 1:1 and the mapping is lossy.
- **`obb.portfolio_intel.risk.metrics` router call.** The router   attempts a dividends fetch that trips on date-parameter shape in some   provider paths; Track A NB03 documents the specific workaround.   Track B computes vol / Sharpe / MaxDD / TE / beta directly from CBOE   returns to keep the free-tier path clean.

## Preview of NB04 (Track B)

Now I know what I own and how concentrated I am. But the picture is static. Events on the calendar (earnings, dividends, splits) and smart-money activity (13F changes, insider transactions) move it every week. NB04 Track B overlays both using SEC + CBOE only.

## 📚 Further reading

All Investopedia links (portfolio, position, weight, diversification, sector breakdown, look-through, HHI, Sharpe, volatility, maximum drawdown, tracking error, benchmark, risk-free rate, correlation) were cited in Track A NB03's Further Reading section — same vocabulary, same links. This notebook does not re-cite them.

**Canonical references** (unchanged from Track A):

- Grinold & Kahn, *Active Portfolio Management*, 2nd ed., ch. 3 —   effective-breadth / Information Ratio.
- Markowitz, "Portfolio Selection," *Journal of Finance* 7(1), 1952 —   the mean-variance foundation of every risk number here.

**Free-authoritative sources used:**

- **SEC EDGAR N-PORT** — quarterly holdings for '40-Act US-registered funds (§3, §4)
- **CBOE EOD** — daily closes for risk math (§6)
